# Evaluation Notebook

This file performs evaluation of the diffusion-based predictor, stores results in a Zarr file, and generates plots.

## Configuration

In [ ]:
# Set the output file.
output_file = "./results/toy_problem.zarr"

## Setup

In [ ]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
import numpy as np
from multiprocessing import Process
from args import *

MAX_PROCESSES = 45

# Set default arguments for evaluation. Feel free to change these!
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(6)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(3)
default_args["episode_length"] = wrap_arg_val(200)
default_args["seed"] = wrap_arg_val(1)

default_args["observation_mask"] = wrap_arg_val(True)
default_args["random_start_positions"] = wrap_arg_val(False)
default_args["diffusion_autoregression_steps"] = wrap_arg_val(1)

In [ ]:
# Evaluation

# Evaluation

In [ ]:
# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")
    # print("Output file already exists. Overwriting evaluation.")

else:
# if True:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)
    
    # obs_probabilities = np.arange(0.0, 1.0, 0.01).tolist()
    obs_probabilities = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

    # Add diffuser series.
    for obs_prob in obs_probabilities:
        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-draft1/wandb/run-20260422_010519-83eoz3ta/files", default_args)
        args["eval_series"] = "Diffusion Predictor"
        args["experiment_name"] = wrap_arg_val(f"obs_prob_{obs_prob:.2f}")
        args["observation_probability"] = wrap_arg_val(obs_prob)
        start_process(args)

        # Add KF series.
        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-draft1/wandb/run-20260422_010519-83eoz3ta/files", default_args)
        args["eval_series"] = "Kalman Filter"
        args["observation_probability"] = wrap_arg_val(obs_prob)
        args["experiment_name"] = wrap_arg_val(f"obs_prob_{obs_prob:.2f}")
        args["prediction_history_window"] = wrap_arg_val(1)
        args["prediction_ensemble_size"] = wrap_arg_val(1)
        args["algorithm_class"] = wrap_arg_val("")
        args["policy_class"] = wrap_arg_val("qmas.baseline_kf.policy.QmasPolicy")
        args["prediction_history_include_actions"] = wrap_arg_val(True)
        start_process(args)

    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

# Analysis

In [ ]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

In [ ]:
%reload_ext autoreload
%autoreload 2
from plots import *
import numpy as np

x_key = 'observation_probability'

# Data for plotting
plots = {
    'tracking_error_mean': {
        'title': 'Tracking Error',
        'ylabel': 'Tracking Error (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['tracking_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['prediction_error_mean'], axis=tuple(range(1, output[series][exp]['prediction_error_mean'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_trajectory_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_mean': {
        'title': 'Prediction Uncertainty',
        'ylabel': 'Prediction Uncertainty (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_mean': {
        'title': 'Prediction Uncertainty Gap',
        'ylabel': 'Prediction Uncertainty Gap (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_min': {
        'title': 'Prediction Uncertainty Gap (Min)',
        'ylabel': 'Prediction Uncertainty Gap Min (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_min']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_max': {
        'title': 'Prediction Uncertainty Gap (Max)',
        'ylabel': 'Prediction Uncertainty Gap Max (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_max']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    }
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in output]

plot(plots, timesteps=timesteps)
